# 05. 사전을 인코더에 넣은 태거 (tagger v2)
03·04 는 사전·태거 점수를 **고정 가중치로 더하기만** 해서, 사전이 '누수'를 1순위로 줘도 문맥상 '뉴스'가 맞는 경우를 가려내지 못했다 (04: 태거를 조금만 섞어도 0.9834 → 0.980).

v2 는 사전의 제안(1·2순위 글자와 확률, '사전에 없음' 확률)을 **입력 특징**으로 넣어, 모델이 문맥을 보고 사전을 따를지 무시할지 학습한다.

- 학습 문장 특징은 **자기 문장을 뺀 사전**(leave-one-out)으로 계산 → 사전을 과하게 믿는 것 방지
- 증강 문장은 미리 `pool`(기본 4) 벌 만들어 특징을 한 번만 계산해 캐시 (`outputs/v2_cache_split.pkl`)
- 검증 분할은 01~04 와 동일 → 02 태거(0.9737)·01(0.9834) 과 바로 비교

**드라이브 끊김 대책:** 드라이브에서 바로 돌리지 않고 Colab 로컬 디스크(`/content/work`)로 복사해서 실행, 단계마다 `outputs/` 만 드라이브에 되돌려 저장한다.

**순서:** 0) 준비 → 1) prep (CPU) → 2) train (GPU, 약 25분) → 3) 비교 → 4) test. **GPU 런타임, 첫 셀부터 실행.**

## 0) 준비: 드라이브 → 로컬 디스크 복사

In [ ]:
import os
os.chdir('/content')  # 끊긴 드라이브 안에 머물러 있으면 모든 명령이 실패하므로 먼저 빠져나옴
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # 끊긴 연결이 남아 있어도 새로 연결
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' "$DRIVE/" /content/work/
%cd /content/work
!ls; nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 결과를 드라이브에 저장 (각 단계 끝난 뒤 실행). 드라이브가 끊겼으면 먼저 재마운트.
import os
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) 사전 특징 캐시 생성 (CPU, 진행률 10% 단위로 출력)
출력의 `[train 실제 입력, leave-one-out]` 과 `[val]` 진단 줄을 비교: 두 줄의 '제안 1순위 정확도'가 비슷해야 학습·검증 조건이 맞는 것.
캐시(수백 MB)는 로컬 디스크에만 두고 드라이브로는 복사하지 않음.

In [ ]:
!python -u tagger_v2.py --mode prep --workers 6

## 2) 학습
**1차 실험(처음부터, pool 증강만)은 0.9606 으로 02(0.9737)보다 낮았다.** 진단 결과:
- 사전 1순위가 틀린 곳을 그대로 따라간 오류 652 → 1,454 (사전 베끼기)
- 사전과 무관한 위치 오류 292 → 867 (pool 4벌 반복으로 과적합)

대책 (기본값으로 반영): 매 epoch **새 노이즈 2개**(사전 특징 없이) + 사전 특징 **30% 확률로 끄기** + **02 체크포인트에서 이어 학습**.

⚠️ `v2_cache_split.pkl` 은 로컬 디스크에만 있으므로 런타임이 바뀌었으면 1) prep 을 다시 실행.

In [ ]:
!python -u tagger_v2.py --mode train --init_ckpt outputs/tagger_best.pt --epochs 15 --lr 2e-4
save_back()

### (비교용) 처음부터 학습 — 새 대책만 적용
`--run_name scratch` 로 파일 이름이 달라서 위 결과를 덮어쓰지 않는다. 주석(`#`)을 지우고 실행.

In [ ]:
!python -u tagger_v2.py --mode train --epochs 30 --run_name scratch
save_back()


### 1차 실험 기록

In [ ]:
# [1차 실험 기록] 처음부터 학습, pool 2벌만 반복 → best 0.9606 (02 의 0.9737 보다 낮음)
# !python -u tagger_v2.py --mode train --epochs 30 --aug 2 --fresh 0 --feat_drop 0


## 3) 비교: 01 / 02 / 03 / v2

In [ ]:
import pandas as pd
from collections import Counter
res = {}
for name, f in [('01 noisy-channel', 'outputs/val_pred.csv'),
                ('02 tagger', 'outputs/val_pred_tagger.csv'),
                ('03 tagger+dict(decoder)', 'outputs/val_pred_tagger_dict.csv'),
                ('05 tagger v2(encoder)', 'outputs/val_pred_tagger_v2.csv'),
                ('05 v2 scratch', 'outputs/val_pred_tagger_v2_scratch.csv')]:
    try:
        d = pd.read_csv(f); d['pred'] = d['pred'].fillna('')
        res[name] = d.set_index('ID'); print(f'{name:24s} F1 {d.f1.mean():.4f}')
    except FileNotFoundError:
        print(f'{name:24s} 파일 없음')
if '05 tagger v2(encoder)' in res:
    a, b, c = res['02 tagger'], res['05 tagger v2(encoder)'], res['01 noisy-channel']
    d = b.f1 - a.f1
    print(f'v2 vs 02: 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}')
    d1 = b.f1 - c.f1
    print(f'v2 vs 01: v2 가 나은 행 {(d1>0).sum()} / 01 이 나은 행 {(d1<0).sum()}')
    err = Counter((x, y) for g, p in zip(b.output, b.pred) for x, y in zip(g, p) if x != y)
    print('v2 오류 top:', err.most_common(20))

In [ ]:
r = pd.read_csv('outputs/tagger_v2_runs.csv')
r[r.run_id == r.run_id.iloc[-1]][['epoch', 'loss', 'val_f1', 'sec', 'aug', 'pool', 'dict_oov', 'init_ckpt']]

## 4) test 예측 → `outputs/submission_tagger_v2.csv`

In [ ]:
!python -u tagger_v2.py --mode predict --workers 6
save_back()
pd.read_csv('outputs/submission_tagger_v2.csv').head()

In [ ]:
from google.colab import runtime
runtime.unassign()